# Độ ổn định theo random seed

Mục tiêu là đo độ phân tán best fitness ở Rastrigin 2, 4, 8 và 12 chiều. Giả thuyết là phân bố thay đổi khi số chiều tăng, dù cùng số thế hệ và evaluations.

In [ ]:
import os
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eho import EHOConfig
from objectives import rastrigin
from experiment_helpers.runner import (
    assert_reproducible, compare_configurations, results_frame, seeds_for,
)
from experiment_helpers.statistics import summarize_experiment_results

MODE = os.getenv("EHO_EXPERIMENT_MODE", "QUICK").upper()
seeds = seeds_for(MODE)
base = EHOConfig(population_size=20, max_generations=50)
print(MODE, seeds)
print(base)
assert_reproducible(rastrigin, base, seeds[0])
print("Kiểm tra tái lập cùng seed đã đạt.")

In [ ]:
from experiment_helpers.plotting import plot_fitness_distribution, plot_runtime_comparison

configs = {f"{dimension}D": EHOConfig(dimension=dimension, population_size=20, max_generations=50)
           for dimension in (2, 4, 8, 12)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
frame["absolute_error"] = frame["fitness"].abs()
tolerance = 1e-3
summary = summarize_experiment_results(frame, tolerance)
display(summary)
display(frame[["group", "seed", "absolute_error", "best_in_bounds", "violating_coordinates"]])
plot_fitness_distribution(frame)
_, axis = plt.subplots()
axis.boxplot([frame.loc[frame.group == label, "fitness"] for label in configs])
axis.set_xticks(range(1, len(configs) + 1), list(configs))
axis.set(xlabel="Số chiều", ylabel="Best fitness cuối")
plot_runtime_comparison(summary)
plt.show()

## Cách đọc và kết luận

SD và phương sai dùng mẫu, ddof = 1. Hộp từ Q1 đến Q3, vạch giữa là median. Râu đến các quan sát trong 1,5 IQR. Điểm xa râu vẫn được giữ. Tolerance là 0,001. Sai số tuyệt đối tính theo cực tiểu bằng 0. Đọc tỷ lệ trong cận và tỷ lệ đạt ngưỡng như hai đại lượng riêng.

In [ ]:
print("SD mẫu theo nhóm:")
print(summary["std"].to_string())
print(f"Nhóm có mean thấp nhất: {summary['mean'].idxmin()}; SD thấp nhất: {summary['std'].idxmin()}.")
print(f"Thời gian trung bình nằm trong [{summary.seconds.min():.4f}, {summary.seconds.max():.4f}] giây.")
print("Kết luận chỉ áp dụng cho cấu hình, ngân sách và tập seed trong bảng.")